# Milestone 2: Get Started

## Task 2: Sign in to the AWS console
Here we used our AWS IAM account to sign in to the AWS console. The working region is set to `eu-west-1` throughout this project.

## Edite Configure files on `pinterest-ec2`:

The file `/home/ubuntu/kafka/etc/kafka/s3-sink.properties` is a configuration file for the Confluent S3 Sink Connector, which is part of Kafka Connect. This connector is used to export data from Apache Kafka topics to Amazon S3 in a structured format (e.g., JSON, Avro). Here we need to configure the following properties:
```
topics=<account_ID>.pin,<account_ID>.geo,<account_ID>.user 
s3.region=<your_s3_region> #Set your AWS region
s3.bucket.name=<your_pinterest_confluent_kafka_connect_s3> #Set your S3 bucket name
format.class=io.confluent.connect.s3.format.json.JsonFormat
```

## Configure our own MySQL database to host our data
We either set up our own local database or use a cloud database service like AWS RDS. Let's discuss both

### Setting up our own database locally

1. For macOS, we can use Homebrew to install MySQL:
    ```
    # Install via Homebrew
    brew install mysql

    # Start MySQL service
    brew services start mysql

    # Secure installation (set root password)
    mysql_secure_installation
    ```
2. Download (DBeaver Community Edition)[https://dbeaver.io]. Install and launch DBeaver.
3. Connect DBeaver to MySQL
    Open DBeaver → **Database** → **New Database Connection**.

    Select **MySQL** → Click **Next**.

    Configure the connection:
    
    **Host**: localhost

    **Port**: 3306

    **Username**: root

    **Password**: Enter the root password you set during MySQL installation.

    **Database**: Leave empty (create a new database later).

    Test the connection → Click **Finish**.

4. Create a New Database

    In DBeaver: Right-click your MySQL connection → **Create** → **Database**.
    Name the database (e.g., `pinterest_data_db`) → Click **OK**.

5. Import the SQL File

    Open the `pinterest_data_db` database in DBeaver.
    Right-click `pinterest_data_db` → Tools → Execute Script (or press Ctrl+Shift+X).
    Select your `pinterest_data_db.sql` file → Click **Start**.
    Wait for the script to execute. Check the Log tab for errors.

6. Verify the Import

    Expand the `pinterest_data_db` database → **Tables**.
    Right-click a table → **View Data** to confirm data exists.

### TODO: Setting up our own database on AWS RDS

## Debugging MySQL Connection in DBeaver

- The error **"Public Key Retrieval is not allowed"** typically occurs when connecting to MySQL 8.0+ with certain security configurations. Here's how to fix it in DBeaver:

    Step 1: Edit Your MySQL Connection in DBeaver

    In DBeaver, right-click your MySQL connection → **Edit Connection**.
    Go to the **Connection Settings** tab.

    Step 2: Allow Public Key Retrieval

    Under the **Driver Properties** tab:

    Search for the property `allowPublicKeyRetrieval`.
    Set its value to `TRUE`.
    This bypasses the public key retrieval restriction for authentication.

Additionally, check [Amazon S3 Source Connector for Confluent Cloud](https://docs.confluent.io/cloud/current/connectors/cc-s3-source.html#using-the-confluent-cli) for using the connector

# Milestone 3: Batch Processing: Configure the EC2 Kafka client

## Task 1: Create a .pem key file locally

If You Still Have the Original `.pem` Key when the EC2 was created, Use the existing key to SSH into the instance:
```
ssh -i "~/.ssh/mykeypair.pem" ubuntu@<public-ec2-ip>
```

If You’ve Lost the Original Key, Create a New Key and Add the New Public Key to the Instance:
1. On your local machine, create a new key file:
   ```
   ssh-keygen -y -f mykeypair.pem
   ```
2. Copy/Extract the Public Key from your new `.pem` file, then connect to the EC2 Instance via EC2 Instance Connect and Edit the `~/.ssh/authorized_keys` file:
    ```
    sudo nano ~/.ssh/authorized_keys
    ```
    Replace the existing public key with the new one (or add it as a new line if you want to keep both keys), save and exit `(Ctrl+O, Ctrl+X)`.

## Task 2: Connect to the EC2 instance

1. Here we use the `pinterest-ec2` instance on AWS for this project
2. `Remote - SSH` extension bug in vscode: 
    
    Shortly after starting up the EC2 instance, you may experience the issue of excessive CPU usage by the 'rg' or/and the 'node' process in VSCode (run `top` to monitor CPU usage on the EC2 instance). This can be resolved by first killing the 'rg' or/and the 'node' processes (`kill -9 <rg_PID> <node_PID>`), then setting `"search.followSymlinks"` to false in VSCode Settings: `Command + , (macOS shorcut)`/`Ctrl + , (Windows shorcut)` -> `Search Settings (Settings.json)` -> add this line `"search.followSymlinks": false`. After that, restart the EC2 instance and ssh to it. See [GitHub Issue #98594](https://github.com/microsoft/vscode/issues/98594) for more information.

3. Setting up Elastic IP on AWS: To retain the same public IP address and DNS name after restarts, use AWS Elastic IP (EIP): Go to `EC2 Dashboard` → `Elastic IPs` → `Allocate Elastic IP address`. Then, select the Elastic IP → `Action` → `Associate Elastic IP address` → Choose your EC2 instance and click `Associate` .
4. Final config in `~/.ssh/config`:
    ```
    ###########################################
    ########### pinterest-ec2 Login ###########
    ###########################################
    Host aws-pinterest-ec2
        HostName <your_aws_ec2_elastic_ip>
        User ubuntu
        IdentityFile ~/.ssh/mykeypair.pem
    ```

## Task 3: Create Kafka Topics on EC2

Find your UserId/Account ID using the AWS CLI:
```
aws sts get-caller-identity
```
Under the "Account" section, you will find your Account ID. Here are our three Kafka topics to create:
```
<account_ID>.pin for the Pinterest posts data
<account_ID>.geo for the post geolocation data
<account_ID>.user for the post user data
```
(where account_ID = your_UserId)

Create a Kafka topic like so:
```
kafka-topics --create \
  --bootstrap-server localhost:9092 \
  --replication-factor 1 \
  --partitions 3 \
  --topic <kafka-test-topic>
```
